# Execution study 2026-09 — what execution costs each running sleeve (AUDIT + policy decision)

Pre-registration: [README.md](README.md) (frozen before any run). Write-up: [findings.md](findings.md).

| id | question |
|---|---|
| E0 | facts (what `btc_1m` is, spot-vs-perp basis) and parity gates on the sleeves' own validated numbers |
| E1 | market-order cost: realised spread and decision-to-fill drift |
| E2 | passive (limit) entry: fill rate, improvement, fill-weighted expectancy, no survivorship |
| E3 | take-profit fills: touch vs trade-through; market-at-touch cost |
| E4 | stop-market slippage under stop_path semantics |
| E5 | conditional adverse selection at signal times vs random times (5 s) |
| E6 | re-cost every sleeve under the measured models; apply the pre-registered rules |

Re-run order: `run_e0_facts.py` → `run_e1_market_cost.py` → `run_e2_passive_entry.py` → `run_e3_tp_fills.py`
→ `run_e4_stop_slippage.py` → `run_e5_conditional_as.py` → `run_e6_recost.py`. All read-only against prod.db.


## E0 — facts and parity gates

In [1]:
%run run_e0_facts.py

(a) btc_1m identity: {'n_hours': 58671, 'median_abs_bp_vs_spot': 0.0, 'median_abs_bp_vs_perp': 4.566778195775889, 'share_exact_vs_spot': 0.9997784254572105, 'verdict': 'SPOT'}


(b) basis bp (perp/spot-1): {'full': {'n': 245638, 'mean': -1.325742826235414, 'sd': 6.322668056706434, 'p95_abs': 10.678601942007647, 'span': ['2019-09-08 17:45', '2026-09-12 05:30']}, 'five_s_year': {'n': 35136, 'mean': -4.505136841822187, 'sd': 1.2488415554580996, 'p95_abs': 6.154753317236528}, 'd15_sd': 2.4512851218584806, 'd15_p95_abs': 3.1625336828313677}


(c) CHENTO_BTC: n=101 no-cost 0.8001 (exp 0.8) | 18bp 0.6853 (exp 0.685) -> PASS


(c) CHENTO_ETH: n=77 no-cost 0.7120 (exp 0.712) | 18bp 0.6219 (exp 0.622) -> PASS


(c) SHORT_SQUEEZE port: {'n': 70, 'win': 0.45714285714285713, 'mean_r': 0.3916288386832697, 'pf': 1.64431729954119, 'n_all': 71, 'triggers_all': 71, 'span': ['2022-05-06 07:30:00+00:00', '2026-05-18 14:15:00+00:00']} -> PASS
(c) SQUEEZE_BULL ledger parity: {'n': 122, 'n_matched': 122, 'max_abs_diff': 2.220446049250313e-16, 'pass_': True}
(c) ADX ledger: {'n_closed': 34, 'reasons': {'ADX<20': 23, 'SL': 11}, 'mean_net_pct': 15.192221953536416}
event counts: {'CHENTO_BTC': 101, 'CHENTO_ETH': 77, 'SHORT_SQUEEZE': 71, 'SQUEEZE_BULL': 122, 'ADX': 34}


## E1 — market-order cost

In [2]:
%run run_e1_market_cost.py

spreads: {'roll_5s': {'n_hours': 8760, 'share_defined': 0.06780821917808219, 'median_half_bp': 0.26040672853395774, 'mean_half_bp': 0.3570768024514161, 'p90_half_bp': 0.7454943921070727, 'cs_5s_median_half_bp': 0.020966414601802843}, 'cs_btc_1m': {'median_half_bp': 0.8564777036508252}, 'cs_eth_1m': {'median_half_bp': 1.1383081310136736}}


CHENTO_BTC entry drift 1m: n=101 {'gap_open_cost_bp': (-0.01, [-0.04, 0.02]), 'at_60s_cost_bp': (-0.55, [-1.78, 0.66]), 'at_120s_cost_bp': (-0.54, [-1.77, 0.66])}
CHENTO_BTC entry drift 5s: n=21 {'at_0s_cost_bp': (-0.0, [-0.0, 0.0]), 'at_30s_cost_bp': (0.37, [-0.83, 1.62]), 'at_60s_cost_bp': (1.5, [-0.1, 3.19])}
CHENTO_BTC tif-exit drift: {'1m': (33, 0.04), '5s': (7, -0.0)}


CHENTO_ETH entry drift 1m: n=77 {'gap_open_cost_bp': (-0.01, [-0.03, 0.0]), 'at_60s_cost_bp': (-0.24, [-2.29, 1.66]), 'at_120s_cost_bp': (-0.23, [-2.29, 1.66])}
CHENTO_ETH tif-exit drift: {'1m': (28, -0.02)}


SHORT_SQUEEZE entry drift 1m: n=71 {'gap_open_cost_bp': (0.03, [-0.04, 0.12]), 'at_60s_cost_bp': (-0.31, [-2.58, 1.97]), 'at_120s_cost_bp': (-0.25, [-2.52, 2.02])}
SHORT_SQUEEZE entry drift 5s: n=17 {'at_0s_cost_bp': (0.0, [-0.0, 0.0]), 'at_30s_cost_bp': (1.9, [-2.11, 5.44]), 'at_60s_cost_bp': (-1.85, [-5.32, 0.71])}
SHORT_SQUEEZE tif-exit drift: {'1m': (11, 0.05), '5s': (0, nan)}


SQUEEZE_BULL entry drift 1m: n=122 {'gap_open_cost_bp': (0.0, [-0.04, 0.05]), 'at_60s_cost_bp': (-2.6, [-4.74, -0.44]), 'at_120s_cost_bp': (-2.6, [-4.75, -0.43])}
SQUEEZE_BULL entry drift 5s: n=12 {'at_0s_cost_bp': (0.0, [0.0, 0.0]), 'at_30s_cost_bp': (-1.68, [-4.46, 0.76]), 'at_60s_cost_bp': (0.1, [-3.68, 3.61])}
SQUEEZE_BULL tif-exit drift: {'1m': (31, -0.01), '5s': (5, -0.0)}
ADX drift: {'1m': {'gap_open_cost_bp': {'n': 28, 'mean': -0.07557971694539332, 'median': 0.0}, 'at_60s_cost_bp': {'n': 28, 'mean': -1.733724256700572, 'median': -1.2442101832673065}, 'exit_at_60s_cost_bp': {'n': 19, 'mean': 1.6383069334216902, 'median': -0.07448101627884188}}}


## E2 — passive entry

In [3]:
%run run_e2_passive_entry.py

ADX entry: {'n': 28, 'market_cost_bp_mean': -0.07557971694539332, 'T60': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}, 'T300': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}, 'T900': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}, 'T3600': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}}



CHENTO_BTC: market n=101 mean +0.800 R (halves +1.301/+0.289)
                         T     rule      fb   fill impr_bp mean_r d_vs_mkt   d_h1   d_h2 delay_s
T60_through_market      60  through  market  0.891    0.03    0.8     -0.0    0.0   -0.0     0.0
T300_through_market    300  through  market  0.931    0.02    0.8      0.0    0.0    0.0     0.0
T900_through_market    900  through  market  0.941    0.02    0.8      0.0    0.0    0.0     0.0
T3600_through_market  3600  through  market   0.98    0.02    0.8      0.0    0.0    0.0     0.0
T60_touch_market        60    touch  market   0.98    0.02    0.8     -0.0    0.0   -0.0     0.0
T300_touch_market      300    touch  market   0.99    0.02    0.8      0.0    0.0    0.0     0.0
T900_touch_market      900    touch  market   0.99    0.02    0.8      0.0    0.0    0.0     0.0
T3600_touch_market    3600    touch  market    1.0    0.02    0.8      0.0    0.0    0.0     0.0
T60_through_skip        60  through    skip  0.891    0.03  0.83


CHENTO_ETH: market n=77 mean +0.653 R (halves +0.979/+0.318)
                         T     rule      fb   fill impr_bp mean_r d_vs_mkt   d_h1   d_h2 delay_s
T60_through_market      60  through  market  0.935    0.03  0.652   -0.001 -0.001 -0.001     0.0
T300_through_market    300  through  market  0.948    0.03  0.651   -0.002 -0.001 -0.003     0.0
T900_through_market    900  through  market  0.987    0.02  0.653   -0.001 -0.001    0.0     0.0
T3600_through_market  3600  through  market    1.0    0.02  0.653      0.0    0.0    0.0     0.0
T60_touch_market        60    touch  market  0.987    0.02  0.653     -0.0    0.0   -0.0     0.0
T300_touch_market      300    touch  market    1.0    0.02  0.653      0.0    0.0    0.0     0.0
T900_touch_market      900    touch  market    1.0    0.02  0.653      0.0    0.0    0.0     0.0
T3600_touch_market    3600    touch  market    1.0    0.02  0.653      0.0    0.0    0.0     0.0
T60_through_skip        60  through    skip  0.935    0.03   0.58


SHORT_SQUEEZE: market n=71 mean +0.481 R (halves +0.389/+0.576)
                         T     rule      fb   fill impr_bp mean_r d_vs_mkt   d_h1   d_h2 delay_s
T60_through_market      60  through  market  0.958    0.03  0.476   -0.005 -0.009    0.0     0.0
T300_through_market    300  through  market  0.986    0.03  0.472   -0.009 -0.018    0.0     0.0
T900_through_market    900  through  market  0.986    0.03  0.471    -0.01 -0.019    0.0     0.0
T3600_through_market  3600  through  market  0.986    0.03  0.475   -0.006 -0.012    0.0     0.0
T60_touch_market        60    touch  market    1.0    0.03  0.481      0.0    0.0    0.0     0.0
T300_touch_market      300    touch  market    1.0    0.03  0.481      0.0    0.0    0.0     0.0
T900_touch_market      900    touch  market    1.0    0.03  0.481      0.0    0.0    0.0     0.0
T3600_touch_market    3600    touch  market    1.0    0.03  0.481      0.0    0.0    0.0     0.0
T60_through_skip        60  through    skip  0.958    0.03  0.


SQUEEZE_BULL: market n=122 mean +0.334 R (halves +0.256/+0.412)
                         T     rule      fb   fill impr_bp mean_r d_vs_mkt   d_h1   d_h2 delay_s
T60_through_market      60  through  market  0.959    0.05  0.333     -0.0   -0.0 -0.001     0.0
T300_through_market    300  through  market  0.967    0.04  0.333   -0.001   -0.0 -0.001     0.0
T900_through_market    900  through  market  0.992    0.04  0.334      0.0    0.0    0.0     0.0
T3600_through_market  3600  through  market  0.992    0.04  0.334      0.0    0.0    0.0     0.0
T60_touch_market        60    touch  market    1.0    0.04  0.334      0.0    0.0    0.0     0.0
T300_touch_market      300    touch  market    1.0    0.04  0.334      0.0    0.0    0.0     0.0
T900_touch_market      900    touch  market    1.0    0.04  0.334      0.0    0.0    0.0     0.0
T3600_touch_market    3600    touch  market    1.0    0.04  0.334      0.0    0.0    0.0     0.0
T60_through_skip        60  through    skip  0.959    0.05  0.

## E3 — take-profit fills

In [4]:
%run run_e3_tp_fills.py

CHENTO_BTC {'1m': {'n_target_hits': 14, 'share_through_tick': 1.0, 'share_through_1bp': 0.929, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': -36.638836795320195, 'ci90': [-78.4644565905197, -0.5754202904707164], 'median': -6.132997707312111}}, '5s': {'n_target_hits': 1, 'share_through_tick': 1.0, 'share_through_1bp': 0.0, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': 6.958462529546677, 'ci90': [6.958462529546677, 6.958462529546677], 'median': 6.958462529546677}}}
   decision: {'resting_tp': True, 'share_through_tick_1m': 1.0}


CHENTO_ETH {'1m': {'n_target_hits': 8, 'share_through_tick': 1.0, 'share_through_1bp': 0.875, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': 0.9992217979752205, 'ci90': [-33.61740896406467, 34.26665553528556], 'median': 11.639736277413224}}}
   decision: {'resting_tp': True, 'share_through_tick_1m': 1.0}


SHORT_SQUEEZE {'1m': {'n_target_hits': 21, 'share_through_tick': 1.0, 'share_through_1bp': 0.952, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': -3.9384972837554506, 'ci90': [-9.416050780449265, -0.05643163439946322], 'median': -2.2213371266024047}}, '5s': {'n_target_hits': 5, 'share_through_tick': 1.0, 'share_through_1bp': 0.6, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': -0.47273687049640845, 'ci90': [-1.5358799429184093, 1.1219777381365925], 'median': -0.2618214518542327}}}
   decision: {'resting_tp': True, 'share_through_tick_1m': 1.0}


SQUEEZE_BULL {'1m': {'n_target_hits': 49, 'share_through_tick': 1.0, 'share_through_1bp': 0.918, 'expected_loss_r_per_target_if_resting': 0.013, 'mean_loss_r_touch_only': 0.153, 'poll_exit_cost_bp': {'mean': -3.4369731242061747, 'ci90': [-7.989244386859049, 0.6667912431290483], 'median': -0.1200857334663512}}, '5s': {'n_target_hits': 5, 'share_through_tick': 1.0, 'share_through_1bp': 0.6, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': -1.3031206223257112, 'ci90': [-2.6533643175041384, -0.23605533587617883], 'median': -0.36131310033666947}}}
   decision: {'resting_tp': True, 'share_through_tick_1m': 1.0}


## E4 — stop slippage

In [5]:
%run run_e4_stop_slippage.py

CHENTO_BTC 1m {'n_stops': 54, 'share_gap_through': 0.0, 'median_stop_dist_bp': 163.288, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': -0.006} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': 0.0} poll: {'mean': -1.94, 'ci90': [-5.92, 1.63], 'median': 0.08, 'p90': 12.96} stressed: {'n': 39, 'mean_slip_pathsem_bp': 0.0, 'calm_mean_slip_pathsem_bp': 0.0}
CHENTO_BTC 5s {'n_stops': 13, 'share_gap_through': 0.0, 'median_stop_dist_bp': 123.158, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': -0.005} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': 0.0} poll: {'mean': 0.15, 'ci90': [-2.38, 2.9], 'median': 0.1, 'p90': 5.74} stressed: None


CHENTO_ETH 1m {'n_stops': 41, 'share_gap_through': 0.0, 'median_stop_dist_bp': 241.474, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': -0.03} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': -0.0} poll: {'mean': -2.84, 'ci90': [-7.0, 1.08], 'median': -0.15, 'p90': 13.8} stressed: {'n': 31, 'mean_slip_pathsem_bp': 0.0, 'calm_mean_slip_pathsem_bp': 0.0}


SHORT_SQUEEZE 1m {'n_stops': 39, 'share_gap_through': 0.0, 'median_stop_dist_bp': 34.554, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': -0.093} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': 0.0} poll: {'mean': -3.74, 'ci90': [-7.09, -0.43], 'median': -0.95, 'p90': 8.53} stressed: {'n': 26, 'mean_slip_pathsem_bp': 0.0, 'calm_mean_slip_pathsem_bp': 0.0}
SHORT_SQUEEZE 5s {'n_stops': 12, 'share_gap_through': 0.0, 'median_stop_dist_bp': 38.958, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': 0.052} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': 0.0} poll: {'mean': 1.66, 'ci90': [0.7, 2.54], 'median': 1.73, 'p90': 4.34} stressed: None


SQUEEZE_BULL 1m {'n_stops': 42, 'share_gap_through': 0.0, 'median_stop_dist_bp': 200.0, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': -0.004} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': 0.0} poll: {'mean': -0.72, 'ci90': [-6.05, 4.47], 'median': -0.54, 'p90': 17.2} stressed: {'n': 35, 'mean_slip_pathsem_bp': 0.0, 'calm_mean_slip_pathsem_bp': 0.0}
SQUEEZE_BULL 5s {'n_stops': 2, 'share_gap_through': 0.0, 'median_stop_dist_bp': 200.0, 'slip_pathsem_in_R': 0.0, 'slip_poll_in_R': -0.028} pathsem: {'mean': 0.0, 'ci90': [0.0, 0.0], 'median': 0.0, 'p90': 0.0, 'max': 0.0} poll: {'mean': -5.67, 'ci90': [-7.22, -4.12], 'median': -5.67, 'p90': -4.43} stressed: None
ADX SL: {'n_sl': 9, 'n_located': 9, 'mean_slip_pathsem_bp': 0.0, 'max_slip_pathsem_bp': 0.0, 'mean_slip_poll_bp': -9.774534645999628, 'share_gap_through': 0.0}


## E5 — conditional adverse selection

In [6]:
%run run_e5_conditional_as.py

CHENTO_BTC_all {'n_signals': 41, 'n_cond_filled': 40, 'n_rand': 820, 'fill_rate_cond': 0.975609756097561, 'fill_rate_rand': 0.9841463414634146}
   markout +60s: cond -0.01 bp [-2.46, 2.25] | random -0.18 [-0.59, 0.22] | diff +0.17 [-2.68, 2.84]
   markout +300s: cond +5.56 bp [0.49, 11.28] | random -0.15 [-0.91, 0.58] | diff +5.71 [-0.1, 12.19]
   markout +900s: cond +11.45 bp [5.1, 17.92] | random -0.66 [-1.87, 0.5] | diff +12.11 [4.6, 19.79]
   markout +3600s: cond +5.05 bp [-4.38, 15.36] | random -1.50 [-4.04, 0.99] | diff +6.55 [-5.37, 19.4]


SHORT_SQUEEZE {'n_signals': 17, 'n_cond_filled': 17, 'n_rand': 340, 'fill_rate_cond': 1.0, 'fill_rate_rand': 0.9764705882352941}
   markout +60s: cond -2.28 bp [-4.86, -0.26] | random +0.62 [-0.07, 1.31] | diff -2.90 [-6.18, -0.19]
   markout +300s: cond -0.48 bp [-8.37, 6.35] | random +2.02 [0.59, 3.47] | diff -2.50 [-11.84, 5.76]
   markout +900s: cond +2.44 bp [-7.04, 12.33] | random +1.01 [-1.51, 3.34] | diff +1.43 [-10.38, 13.84]
   markout +3600s: cond +6.86 bp [-19.39, 35.53] | random +2.58 [-2.59, 7.6] | diff +4.29 [-26.99, 38.12]


SQUEEZE_BULL_ALL {'n_signals': 61, 'n_cond_filled': 61, 'n_rand': 1220, 'fill_rate_cond': 1.0, 'fill_rate_rand': 0.9606557377049181}
   markout +60s: cond -0.24 bp [-2.46, 2.25] | random -0.16 [-0.53, 0.21] | diff -0.08 [-2.67, 2.78]
   markout +300s: cond +1.92 bp [-1.62, 5.65] | random +0.11 [-0.66, 0.88] | diff +1.81 [-2.49, 6.31]
   markout +900s: cond +4.39 bp [-2.47, 10.91] | random +0.66 [-0.84, 2.23] | diff +3.73 [-4.69, 11.76]
   markout +3600s: cond +4.68 bp [-7.54, 16.32] | random -1.11 [-3.48, 1.38] | diff +5.79 [-8.92, 19.8]


## E6 — re-cost and decide

In [7]:
%run run_e6_recost.py


CHENTO_BTC: n=101 exits={'stop': 54, 'tif': 33, 'target': 14} gross +0.800 R | drift -0.55 bp | stop slip 0.00 bp
                  mean_bp_per_trade mean_cost_r net_mean_r cost_share_of_gross   net_mar net_first_half net_second_half fill_rate
M0_sleeve                      18.0    0.114814   0.684951            0.143559  0.927585       1.200093        0.159507       1.0
M0_research                    18.0    0.114814   0.684951            0.143559  0.927585       1.200093        0.159507       1.0
M1_measured_taker            9.5931    0.060447   0.739318            0.075581  1.079837       1.249646        0.218782       1.0
M2_hybrid_touch            7.760396    0.048453   0.761231            0.059842  1.241493       1.259483        0.253014  0.990099
M2_hybrid_through          7.382178    0.045549   0.744333            0.057666  1.501031       1.280102        0.197848  0.940594
M3_maker_touch             7.008911    0.044843   0.764841            0.055383  1.254825       1.262722  


CHENTO_ETH: n=77 exits={'stop': 41, 'tif': 28, 'target': 8} gross +0.653 R | drift -0.24 bp | stop slip 0.00 bp
                  mean_bp_per_trade mean_cost_r net_mean_r cost_share_of_gross   net_mar net_first_half net_second_half fill_rate
M0_sleeve                      18.0    0.090057   0.563109            0.137878   0.80703       0.871237        0.246873       1.0
M0_research                    18.0    0.090057   0.563109            0.137878   0.80703       0.871237        0.246873       1.0
M1_measured_taker         10.017539     0.04864   0.604527            0.074468  0.898148       0.922711         0.27797       1.0
M2_hybrid_touch            7.957143    0.038331   0.614851            0.058684  0.921517       0.935109        0.286166       1.0
M2_hybrid_through          7.849351    0.038053   0.596273             0.05999  0.893673       0.898429        0.286166  0.987013
M3_maker_touch             7.120779    0.035335   0.617848            0.054097  0.927263       0.937907    


SHORT_SQUEEZE: n=71 exits={'stop': 39, 'target': 21, 'tif': 11} gross +0.481 R | drift -0.31 bp | stop slip 0.00 bp
                  mean_bp_per_trade mean_cost_r net_mean_r cost_share_of_gross   net_mar net_first_half net_second_half fill_rate
M0_sleeve                      25.0    0.797047  -0.316252            1.657769 -0.209323      -0.282537        -0.35093       1.0
M0_research                     4.0    0.127527   0.353267            0.265243  0.821024       0.281064        0.427534       1.0
M1_measured_taker          9.316547    0.288767   0.192027            0.600604  0.398766       0.135538         0.25013       1.0
M2_hybrid_touch            7.323944    0.225406   0.255757            0.468461  0.549452       0.189416        0.323993       1.0
M2_hybrid_through          7.207042    0.223476   0.247111            0.474888  0.530877       0.172365        0.323993  0.985915
M3_maker_touch             6.967606    0.217194   0.263968            0.451395  0.567093       0.201247


SQUEEZE_BULL: n=122 exits={'target': 49, 'stop': 42, 'tif': 31} gross +0.334 R | drift -2.60 bp | stop slip 0.00 bp
                  mean_bp_per_trade mean_cost_r net_mean_r cost_share_of_gross   net_mar net_first_half net_second_half fill_rate
M0_sleeve                      18.0        0.09   0.243924            0.269522  1.236159       0.166165        0.321684       1.0
M0_research                    18.0        0.09   0.243924            0.269522  1.236159       0.166165        0.321684       1.0
M1_measured_taker          6.672008     0.03336   0.300564            0.099903  1.661303       0.222399        0.378729       1.0
M2_hybrid_touch             6.97459    0.034873    0.29911            0.104415  1.648752       0.221003        0.377217       1.0
M2_hybrid_through          6.906557    0.034533   0.307647             0.10092  1.695809       0.238076        0.377217  0.991803
M3_maker_touch             6.390164    0.031951   0.302032            0.095666  1.680387       0.223642

## Decisions (from results/e6_recost.json)

In [8]:
import json, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
E6 = json.load(open('results/e6_recost.json', encoding='utf-8'))
for sl, d in E6['decisions'].items():
    print(sl)
    print('  rule 1 cost constant :', {k: (round(v, 2) if isinstance(v, float) else v) for k, v in d['rule1'].items()})
    print('  rule 2 maker entry   :', d['rule2_entry'])
    print('  rule 3 resting TP    :', d['rule3_tp'])
    print('  rule 4 viability     :', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d['rule4'].items()})
print('ADX  :', E6['ADX']); print('CARRY:', E6['CARRY'])


CHENTO_BTC
  rule 1 cost constant : {'coded_sleeve_bp': 18.0, 'measured_taker_bp': 9.59, 'measured_ci90': [8.35769630354961, 10.800176628296947], 'differs_by_gt_3bp': True, 'ci_excludes_coded': True, 'recommend_change': True}
  rule 2 maker entry   : {'maker_entry': False, 'cell': None, 'detail': None, 'fill_rate_T300_1m_on_5s_events': 1.0, 'fill_rate_T300_5s': 1.0, 'resolution_check_pass': True}
  rule 3 resting TP    : {'resting_tp': True, 'share_through_tick_1m': 1.0}
  rule 4 viability     : {'m1_cost_share': 0.076, 'm2_through_cost_share': 0.058, 'm2_touch_cost_share': 0.06, 'verdict': 'VIABLE'}
CHENTO_ETH
  rule 1 cost constant : {'coded_sleeve_bp': 18.0, 'measured_taker_bp': 10.02, 'measured_ci90': [7.9691159802287626, 11.912417329132339], 'differs_by_gt_3bp': True, 'ci_excludes_coded': True, 'recommend_change': True}
  rule 2 maker entry   : {'maker_entry': False, 'cell': None, 'detail': None}
  rule 3 resting TP    : {'resting_tp': True, 'share_through_tick_1m': 1.0}
  rule 4 

## E6 tables

In [9]:
for sl, d in E6['sleeves'].items():
    print(f"\n{sl}: n={d['n']} exits={d['exit_kinds']} gross {d['gross']['mean_r']:+.3f} R  drift {d['drift_bp']:+.2f} bp  stop slip {d['stop_slip_bp']:.2f} bp")
    display(pd.DataFrame(d['models']).T[['mean_bp_per_trade', 'mean_cost_r', 'net_mean_r', 'cost_share_of_gross', 'net_mar', 'net_first_half', 'net_second_half', 'fill_rate']].round(3))



CHENTO_BTC: n=101 exits={'stop': 54, 'tif': 33, 'target': 14} gross +0.800 R  drift -0.55 bp  stop slip 0.00 bp


,mean_bp_per_trade,mean_cost_r,net_mean_r,cost_share_of_gross,net_mar,net_first_half,net_second_half,fill_rate
M0_sleeve,18.0,0.114814,0.684951,0.143559,0.927585,1.200093,0.159507,1.0
M0_research,18.0,0.114814,0.684951,0.143559,0.927585,1.200093,0.159507,1.0
M1_measured_taker,9.5931,0.060447,0.739318,0.075581,1.079837,1.249646,0.218782,1.0
M2_hybrid_touch,7.760396,0.048453,0.761231,0.059842,1.241493,1.259483,0.253014,0.990099
M2_hybrid_through,7.382178,0.045549,0.744333,0.057666,1.501031,1.280102,0.197848,0.940594
M3_maker_touch,7.008911,0.044843,0.764841,0.055383,1.254825,1.262722,0.257002,0.990099
M3_maker_through,6.630693,0.041939,0.747943,0.053095,1.519458,1.283342,0.201836,0.940594



CHENTO_ETH: n=77 exits={'stop': 41, 'tif': 28, 'target': 8} gross +0.653 R  drift -0.24 bp  stop slip 0.00 bp


,mean_bp_per_trade,mean_cost_r,net_mean_r,cost_share_of_gross,net_mar,net_first_half,net_second_half,fill_rate
M0_sleeve,18.0,0.090057,0.563109,0.137878,0.80703,0.871237,0.246873,1.0
M0_research,18.0,0.090057,0.563109,0.137878,0.80703,0.871237,0.246873,1.0
M1_measured_taker,10.017539,0.04864,0.604527,0.074468,0.898148,0.922711,0.27797,1.0
M2_hybrid_touch,7.957143,0.038331,0.614851,0.058684,0.921517,0.935109,0.286166,1.0
M2_hybrid_through,7.849351,0.038053,0.596273,0.05999,0.893673,0.898429,0.286166,0.987013
M3_maker_touch,7.120779,0.035335,0.617848,0.054097,0.927263,0.937907,0.289367,1.0
M3_maker_through,7.042857,0.035134,0.599193,0.055387,0.899265,0.901074,0.289367,0.987013



SHORT_SQUEEZE: n=71 exits={'stop': 39, 'target': 21, 'tif': 11} gross +0.481 R  drift -0.31 bp  stop slip 0.00 bp


,mean_bp_per_trade,mean_cost_r,net_mean_r,cost_share_of_gross,net_mar,net_first_half,net_second_half,fill_rate
M0_sleeve,25.0,0.797047,-0.316252,1.657769,-0.209323,-0.282537,-0.35093,1.0
M0_research,4.0,0.127527,0.353267,0.265243,0.821024,0.281064,0.427534,1.0
M1_measured_taker,9.316547,0.288767,0.192027,0.600604,0.398766,0.135538,0.25013,1.0
M2_hybrid_touch,7.323944,0.225406,0.255757,0.468461,0.549452,0.189416,0.323993,1.0
M2_hybrid_through,7.207042,0.223476,0.247111,0.474888,0.530877,0.172365,0.323993,0.985915
M3_maker_touch,6.967606,0.217194,0.263968,0.451395,0.567093,0.201247,0.328482,1.0
M3_maker_through,6.883099,0.215799,0.254788,0.458575,0.54737,0.183141,0.328482,0.985915



SQUEEZE_BULL: n=122 exits={'target': 49, 'stop': 42, 'tif': 31} gross +0.334 R  drift -2.60 bp  stop slip 0.00 bp


,mean_bp_per_trade,mean_cost_r,net_mean_r,cost_share_of_gross,net_mar,net_first_half,net_second_half,fill_rate
M0_sleeve,18.0,0.09,0.243924,0.269522,1.236159,0.166165,0.321684,1.0
M0_research,18.0,0.09,0.243924,0.269522,1.236159,0.166165,0.321684,1.0
M1_measured_taker,6.672008,0.03336,0.300564,0.099903,1.661303,0.222399,0.378729,1.0
M2_hybrid_touch,6.97459,0.034873,0.29911,0.104415,1.648752,0.221003,0.377217,1.0
M2_hybrid_through,6.906557,0.034533,0.307647,0.10092,1.695809,0.238076,0.377217,0.991803
M3_maker_touch,6.390164,0.031951,0.302032,0.095666,1.680387,0.223642,0.380422,1.0
M3_maker_through,6.322131,0.031611,0.310569,0.09238,1.727883,0.240716,0.380422,0.991803


## E0 / E1 tables

In [10]:
E0 = json.load(open('results/e0_facts.json', encoding='utf-8'))
print('btc_1m identity:', E0['btc_1m_identity']); print('basis bp:', json.dumps({k: v for k, v in E0['basis_bp'].items() if k != 'by_year'}, indent=1))
print('gates:', json.dumps(E0['gates'], indent=1, default=str)[:3000])
E1 = json.load(open('results/e1_market_cost.json', encoding='utf-8'))
print('spreads:', json.dumps({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, dict)} for k, v in E1['spreads'].items()}, indent=1))
rows = []
for sl, d in E1['entry_drift'].items():
    for res, dd in d.items():
        for k, v in dd.items():
            if isinstance(v, dict) and 'mean' in v:
                rows.append(dict(sleeve=sl, res=res, measure=k, n=dd.get('n'), mean=v['mean'], ci90=v.get('ci90'), median=v.get('median')))
display(pd.DataFrame(rows).round(2))


btc_1m identity: {'n_hours': 58671, 'median_abs_bp_vs_spot': 0.0, 'median_abs_bp_vs_perp': 4.566778195775889, 'share_exact_vs_spot': 0.9997784254572105, 'verdict': 'SPOT'}
basis bp: {
 "full": {
  "n": 245638,
  "mean": -1.325742826235414,
  "sd": 6.322668056706434,
  "p95_abs": 10.678601942007647,
  "span": [
   "2019-09-08 17:45",
   "2026-09-12 05:30"
  ]
 },
 "five_s_year": {
  "n": 35136,
  "mean": -4.505136841822187,
  "sd": 1.2488415554580996,
  "p95_abs": 6.154753317236528
 },
 "d15_sd": 2.4512851218584806,
 "d15_p95_abs": 3.1625336828313677
}
gates: {
 "CHENTO_BTC": {
  "n": 101,
  "mean_r_nocost": 0.8001439689433522,
  "mean_r_18bp": 0.6853304073773573,
  "expected": {
   "nocost": 0.8,
   "cost": 0.685
  },
  "pass_": true,
  "spot_1m_vs_perp_close_bp": {
   "median_abs": 4.456232200018562,
   "p95_abs": 8.339909510380883
  }
 },
 "CHENTO_ETH": {
  "n": 77,
  "mean_r_nocost": 0.7119886757982192,
  "mean_r_18bp": 0.6219311868939733,
  "expected": {
   "nocost": 0.712,
   "cos

,sleeve,res,measure,n,mean,ci90,median
0,CHENTO_BTC,1m,gap_open_cost_bp,101.0,-0.01,"[-0.03603169276552024, 0.016684506318314245]",0.00
1,CHENTO_BTC,1m,at_60s_cost_bp,101.0,-0.55,"[-1.7848779538761315, 0.6576023708712067]",-0.52
2,CHENTO_BTC,1m,at_120s_cost_bp,101.0,-0.54,"[-1.7729635992314325, 0.6557588274330529]",-0.52
3,CHENTO_BTC,1m,stressed_top5pct_at_60s_cost_bp,101.0,-5.18,None,NaN
4,CHENTO_BTC,5s,at_0s_cost_bp,21.0,-0.00,"[-0.0005194281677614455, 0.0002096622419924141]",0.00
5,CHENTO_BTC,5s,at_30s_cost_bp,21.0,0.37,"[-0.8299247640834488, 1.619354951595489]",-0.64
6,CHENTO_BTC,5s,at_60s_cost_bp,21.0,1.50,"[-0.10345915163309817, 3.193542907716653]",1.68
7,CHENTO_ETH,1m,gap_open_cost_bp,77.0,-0.01,"[-0.03332736013804225, 0.003168096477466018]",0.00
8,CHENTO_ETH,1m,at_60s_cost_bp,77.0,-0.24,"[-2.288026876914095, 1.6552744719894799]",-0.11
9,CHENTO_ETH,1m,at_120s_cost_bp,77.0,-0.23,"[-2.292843050516656, 1.6592156876863793]",-0.11


## E2 tables

In [11]:
E2 = json.load(open('results/e2_passive_entry.json', encoding='utf-8'))
for sl, d in E2['sleeves'].items():
    s = d['1m']; m = s['market']
    print(f"\n{sl}: market n={m['n']} mean {m['mean_r']:+.3f} R (halves {m['first_half_mean']:+.3f}/{m['second_half_mean']:+.3f})")
    display(pd.DataFrame({c: dict(T=v['T'], rule=v['rule'], fb=v['fallback'], fill=v['fill_rate'], impr_bp=v['mean_impr_bp_filled'], mean_r=v['mean_r'],
                                  d_vs_mkt=v['delta_vs_market'], d_h1=v['delta_first_half'], d_h2=v['delta_second_half']) for c, v in s.items() if c != 'market'}).T.round(3))
    print('decision:', E2['decision'][sl])
print('ADX:', E2['adx'])



CHENTO_BTC: market n=101 mean +0.800 R (halves +1.301/+0.289)


,T,rule,fb,fill,impr_bp,mean_r,d_vs_mkt,d_h1,d_h2
T300_through_market,300,through,market,0.930693,0.023946,0.799783,0.000001,0.000001,0.0
T300_through_skip,300,through,skip,0.930693,0.023946,0.799783,0.000001,0.039217,-0.04
T300_touch_market,300,touch,market,0.990099,0.02251,0.799783,0.000001,0.000001,0.0
T300_touch_skip,300,touch,skip,0.990099,0.02251,0.809684,0.009902,0.000001,0.02
T3600_through_market,3600,through,market,0.980198,0.022737,0.799783,0.000001,0.000001,0.0
T3600_through_skip,3600,through,skip,0.980198,0.022737,0.819585,0.019803,0.000001,0.04
T3600_touch_market,3600,touch,market,1.0,0.022287,0.799783,0.000001,0.000001,0.0
T3600_touch_skip,3600,touch,skip,1.0,0.022287,0.799783,0.000001,0.000001,0.0
T60_through_market,60,through,market,0.891089,0.025011,0.79955,-0.000233,0.000001,-0.000471
T60_through_skip,60,through,skip,0.891089,0.025011,0.830683,0.030901,0.058824,0.002419


decision: {'maker_entry': False, 'cell': None, 'detail': None, 'fill_rate_T300_1m_on_5s_events': 1.0, 'fill_rate_T300_5s': 1.0, 'resolution_check_pass': True}

CHENTO_ETH: market n=77 mean +0.653 R (halves +0.979/+0.318)


,T,rule,fb,fill,impr_bp,mean_r,d_vs_mkt,d_h1,d_h2
T300_through_market,300,through,market,0.948052,0.025667,0.650868,-0.0023,-0.00133,-0.003296
T300_through_skip,300,through,skip,0.948052,0.025667,0.626221,-0.026947,-0.011574,-0.042726
T300_touch_market,300,touch,market,1.0,0.024333,0.653183,0.000015,0.000015,0.000014
T300_touch_skip,300,touch,skip,1.0,0.024333,0.653183,0.000015,0.000015,0.000014
T3600_through_market,3600,through,market,1.0,0.024333,0.653183,0.000015,0.000015,0.000014
T3600_through_skip,3600,through,skip,1.0,0.024333,0.653183,0.000015,0.000015,0.000014
T3600_touch_market,3600,touch,market,1.0,0.024333,0.653183,0.000015,0.000015,0.000014
T3600_touch_skip,3600,touch,skip,1.0,0.024333,0.653183,0.000015,0.000015,0.000014
T60_through_market,60,through,market,0.935065,0.026023,0.652174,-0.000994,-0.000552,-0.001448
T60_through_skip,60,through,skip,0.935065,0.026023,0.580163,-0.073005,-0.011574,-0.136053


decision: {'maker_entry': False, 'cell': None, 'detail': None}

SHORT_SQUEEZE: market n=71 mean +0.481 R (halves +0.389/+0.576)


,T,rule,fb,fill,impr_bp,mean_r,d_vs_mkt,d_h1,d_h2
T300_through_market,300,through,market,0.985915,0.032573,0.47198,-0.009181,-0.018107,0.0
T300_through_skip,300,through,skip,0.985915,0.032573,0.470587,-0.010574,-0.020855,0.0
T300_touch_market,300,touch,market,1.0,0.032115,0.481163,0.000001,0.000002,0.0
T300_touch_skip,300,touch,skip,1.0,0.032115,0.481163,0.000001,0.000002,0.0
T3600_through_market,3600,through,market,0.985915,0.032573,0.474994,-0.006167,-0.012163,0.0
T3600_through_skip,3600,through,skip,0.985915,0.032573,0.470587,-0.010574,-0.020855,0.0
T3600_touch_market,3600,touch,market,1.0,0.032115,0.481163,0.000001,0.000002,0.0
T3600_touch_skip,3600,touch,skip,1.0,0.032115,0.481163,0.000001,0.000002,0.0
T60_through_market,60,through,market,0.957746,0.033531,0.476385,-0.004776,-0.00942,0.0
T60_through_skip,60,through,skip,0.957746,0.033531,0.442418,-0.038743,-0.020855,-0.057143


decision: {'maker_entry': False, 'cell': None, 'detail': None, 'fill_rate_T300_1m_on_5s_events': 1.0, 'fill_rate_T300_5s': 1.0, 'resolution_check_pass': True}

SQUEEZE_BULL: market n=122 mean +0.334 R (halves +0.256/+0.412)


,T,rule,fb,fill,impr_bp,mean_r,d_vs_mkt,d_h1,d_h2
T300_through_market,300,through,market,0.967213,0.044678,0.333302,-0.000591,-0.000209,-0.000974
T300_through_skip,300,through,skip,0.967213,0.044678,0.342392,0.008498,0.021064,-0.004067
T300_touch_market,300,touch,market,1.0,0.043213,0.333983,0.000089,0.000178,0.0
T300_touch_skip,300,touch,skip,1.0,0.043213,0.333983,0.000089,0.000178,0.0
T3600_through_market,3600,through,market,0.991803,0.04357,0.333983,0.000089,0.000178,0.0
T3600_through_skip,3600,through,skip,0.991803,0.04357,0.34218,0.008286,0.016571,0.0
T3600_touch_market,3600,touch,market,1.0,0.043213,0.333983,0.000089,0.000178,0.0
T3600_touch_skip,3600,touch,skip,1.0,0.043213,0.333983,0.000089,0.000178,0.0
T60_through_market,60,through,market,0.959016,0.04506,0.333409,-0.000484,-0.000055,-0.000913
T60_through_skip,60,through,skip,0.959016,0.04506,0.350588,0.016695,0.037457,-0.004067


decision: {'maker_entry': False, 'cell': None, 'detail': None, 'fill_rate_T300_1m_on_5s_events': 1.0, 'fill_rate_T300_5s': 1.0, 'resolution_check_pass': True}
ADX: {'n': 28, 'market_cost_bp_mean': -0.07557971694539332, 'T60': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}, 'T300': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}, 'T900': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}, 'T3600': {'fill_rate': 1.0, 'mean_impr_bp_filled': 0.07930167173013601}}


## E3 / E4 / E5 tables

In [12]:
E3 = json.load(open('results/e3_tp_fills.json', encoding='utf-8'))
for sl, d in E3['sleeves'].items():
    print(sl, {res: {k: (round(v, 3) if isinstance(v, float) else v) for k, v in dd.items() if k not in ('unfilled_alt_kinds',)} for res, dd in d.items()})
E4 = json.load(open('results/e4_stop_slippage.json', encoding='utf-8'))
for sl, d in E4['sleeves'].items():
    for res, dd in d.items():
        print(sl, res, 'n', dd.get('n_stops'), 'gap-through', dd.get('share_gap_through'), 'median stop bp', round(dd.get('median_stop_dist_bp', float('nan')), 1),
              'pathsem', {k: (round(v, 2) if isinstance(v, float) else v) for k, v in dd.get('slip_pathsem_bp', {}).items()},
              'poll', {k: (round(v, 2) if isinstance(v, float) else v) for k, v in dd.get('slip_poll_bp', {}).items()})
print('ADX SL:', {k: v for k, v in E4['adx'].items() if k != 'rows'})
E5 = json.load(open('results/e5_conditional_as.json', encoding='utf-8'))
for name, d in E5['sets'].items():
    print(name, {k: v for k, v in d.items() if not isinstance(v, dict)})
    for k in (60, 300, 900, 3600):
        m = d.get(f'mo_{k}')
        if m: print(f"   +{k}s: cond {m['cond_mean']:+.2f} {[round(x, 2) for x in m['cond_ci90']]} | random {m['rand_mean']:+.2f} | diff {m['diff']:+.2f} {[round(x, 2) for x in m['diff_ci90']]}")


CHENTO_BTC {'1m': {'n_target_hits': 14, 'share_through_tick': 1.0, 'share_through_1bp': 0.929, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': -36.638836795320195, 'ci90': [-78.4644565905197, -0.5754202904707164], 'median': -6.132997707312111}}, '5s': {'n_target_hits': 1, 'share_through_tick': 1.0, 'share_through_1bp': 0.0, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': 6.958462529546677, 'ci90': [6.958462529546677, 6.958462529546677], 'median': 6.958462529546677}}}
CHENTO_ETH {'1m': {'n_target_hits': 8, 'share_through_tick': 1.0, 'share_through_1bp': 0.875, 'expected_loss_r_per_target_if_resting': 0.0, 'mean_loss_r_touch_only': 0.0, 'poll_exit_cost_bp': {'mean': 0.9992217979752205, 'ci90': [-33.61740896406467, 34.26665553528556], 'median': 11.639736277413224}}}
SHORT_SQUEEZE {'1m': {'n_target_hits': 21, 'share_through_tick': 1.0, 'share_through_1bp': 0.952, 'expected_